# Notebook: Perfis de Vulnerabilidade Severa de Hortolândia (PVSE)

**id_rastreabilidade:** RTB_006  
**Versão:** v01  
**Data de criação:** 09/05/2026  
**Última atualização:** 09/05/2026  
**Responsável:** Ailton Vendramini  

---

## 🎯 Finalidade
Calcular perfis de vulnerabilidade severa multidimensional a partir da base limpa do CadÚnico.
Esta camada é **complementar ao IVS-H** — não substitui nem altera os indicadores clássicos.

---

## 📥 Fonte de Dados
**Fonte:** CadÚnico  
**Camada de entrada:** `02_limpo`  
**Camada de saída:** `outputs/tabelas/cadunico/`  
**Período de referência da execução:** `2025_12`  
**Tipo de recorte temporal:** corte transversal  

**Arquivo de entrada:**  
`C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\cadunico\\02_limpo\\2025_12\\cadunico_limpo.csv`

---

## 🧱 Base Conceitual
- `00_governanca/ivs_vs_ivsh_comparativo_v09.md`
- `05_calculo_ivsh_cadunico_v02.ipynb` — indicadores IVS-H de referência

---

## ⚙️ Escopo técnico deste notebook

**Este notebook deve:**
1. Importações
2. Carregamento da base limpa
3. Validação de colunas
4. PVSE_A — Exclusão Produtiva Analfabeta
5. PVSE_B — Maternidade Vulnerável Severa
6. PVSE_C — Idoso Desprotegido
7. Síntese — painel PVSE

**Este notebook NÃO deve:**
- recalcular variáveis do IVS-H clássico (RT_01, RT_04, CH_05, CH_06, CH_07)
- identificar indivíduos — análise sempre agregada por loteamento
- alterar arquivos da camada `01_bruto` ou `02_limpo`

---

## 📊 Outputs esperados

| tipo_output | descrição | destino |
|---|---|---|
| analitico | contagens e percentuais por perfil | `outputs/tabelas/cadunico/` |
| territorial | distribuição por loteamento (proxy) | `outputs/tabelas/cadunico/` |

---

## 🧠 Observações
- Os perfis PVSE **não substituem** o IVS-H — são uma camada derivada para intervenção direta.
- Os resultados devem ser usados de forma **agregada** — nunca para identificação individual.
- A separação metodológica preserva a comparabilidade nacional do IVS-H.

---

## 🔥 Perfis definidos

| Código | Nome | Conceito |
|---|---|---|
| PVSE_A | Exclusão Produtiva Analfabeta | analfabetismo + ausência de remuneração (15–59 anos) |
| PVSE_B | Maternidade Vulnerável Severa | mães analfabetas + ausência de pensão alimentícia |
| PVSE_C | Idoso Desprotegido | idoso analfabeto + pobre + sem aposentadoria + sem BPC/BF |

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import date

## 2. Carregamento da base limpa

In [ ]:
# constantes de referência
PERIODO_REFERENCIA = '2025_12'
MEIO_SM            = 759.00  # meio salário mínimo em dez/2025

# caminho absoluto da base limpa gerada pelo notebook 02
caminho_entrada = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\02_limpo\2025_12\cadunico_limpo.csv'

df = pd.read_csv(caminho_entrada, sep=';', encoding='utf-8', low_memory=False)

print(f'Base carregada: {df.shape[0]} linhas, {df.shape[1]} colunas')
print(f'Período de referência: {PERIODO_REFERENCIA}')

## 3. Validação de colunas

In [ ]:
# colunas necessárias para os três perfis PVSE
colunas_necessarias = [
    'idade',                         # base para todos os perfis
    'p.cod_sabe_ler_escrever_memb',  # analfabetismo
    'p.cod_sexo_pessoa',             # PVSE_B
    'p.fx_renda_individual_805',     # PVSE_A — remuneração trabalho
    'p.fx_renda_individual_809_2',   # PVSE_C — aposentadoria
    'p.fx_renda_individual_809_4',   # PVSE_B — pensão alimentícia
    'd.vlr_renda_media_fam',         # PVSE_C — renda familiar
    'd.marc_pbf',                    # PVSE_C — BPC/Bolsa Família
    'd.nom_localidade_fam'           # proxy territorial
]

ausentes = [col for col in colunas_necessarias if col not in df.columns]

if ausentes:
    print(f'ATENÇÃO — colunas ausentes: {ausentes}')
else:
    print('Todas as colunas necessárias estão presentes')

## 4. PVSE_A — Exclusão Produtiva Analfabeta

In [ ]:
# PVSE_A: pessoas de 15 a 59 anos que são analfabetas E não têm remuneração do trabalho principal
# p.fx_renda_individual_805 == 0 → sem remuneração do trabalho principal
# conceito: exclusão estrutural — sem escolaridade e sem inserção produtiva

mask_15_59      = (df['idade'] >= 15) & (df['idade'] < 60)
mask_analfabeto = df['p.cod_sabe_ler_escrever_memb'] == 2
mask_sem_renda  = df['p.fx_renda_individual_805'] == 0

pvse_a = (mask_15_59 & mask_analfabeto & mask_sem_renda).sum()
base_a = (mask_15_59 & mask_analfabeto).sum()  # analfabetos 15–59 (base de referência)

print(f'PVSE_A — Exclusão Produtiva Analfabeta')
print(f'Analfabetos 15–59 (base)      : {base_a}')
print(f'Sem remuneração de trabalho   : {pvse_a}')
print(f'% dos analfabetos 15–59       : {(pvse_a / base_a * 100).round(1)}%')

## 5. PVSE_B — Maternidade Vulnerável Severa

In [ ]:
# PVSE_B: mulheres analfabetas que não recebem pensão alimentícia
# p.fx_renda_individual_809_4 == 0 → sem pensão alimentícia
# conceito: vulnerabilidade intergeracional — sem escolaridade e sem suporte econômico externo

mask_feminino   = df['p.cod_sexo_pessoa'] == 2
mask_sem_pensao = df['p.fx_renda_individual_809_4'] == 0

pvse_b = (mask_analfabeto & mask_feminino & mask_sem_pensao).sum()
base_b = (mask_analfabeto & mask_feminino).sum()  # mulheres analfabetas (base)

print(f'PVSE_B — Maternidade Vulnerável Severa')
print(f'Mulheres analfabetas (base)   : {base_b}')
print(f'Sem pensão alimentícia        : {pvse_b}')
print(f'% das mulheres analfabetas    : {(pvse_b / base_b * 100).round(1)}%')

## 6. PVSE_C — Idoso Desprotegido

In [ ]:
# PVSE_C: idosos analfabetos + renda familiar ≤ ½ SM + sem aposentadoria + sem BPC/BF
# p.fx_renda_individual_809_2 == 0 → sem aposentadoria
# d.marc_pbf == 0               → sem BPC nem Bolsa Família
# conceito: invisibilidade institucional total — idoso pobre fora de qualquer rede de proteção

mask_idoso           = df['idade'] >= 60
mask_renda_baixa     = df['d.vlr_renda_media_fam'] <= MEIO_SM
mask_sem_aposent     = df['p.fx_renda_individual_809_2'] == 0
mask_nao_pbf         = df['d.marc_pbf'] == 0

mask_idoso_analfabeto = mask_idoso & mask_analfabeto

pvse_c = (mask_idoso_analfabeto & mask_renda_baixa & mask_sem_aposent & mask_nao_pbf).sum()
base_c = mask_idoso_analfabeto.sum()  # idosos analfabetos (base)

print(f'PVSE_C — Idoso Desprotegido')
print(f'Idosos analfabetos (base)         : {base_c}')
print(f'Pobres + sem aposentadoria + sem BPC/BF: {pvse_c}')
print(f'% dos idosos analfabetos           : {(pvse_c / base_c * 100).round(1)}%')

## 7. Síntese — Painel PVSE

In [ ]:
# resumo consolidado dos perfis de vulnerabilidade severa
print('=== PAINEL PVSE — PERFIS DE VULNERABILIDADE SEVERA ===')
print(f'Período de referência: {PERIODO_REFERENCIA}')
print(f'Base: CadÚnico dez/2025 — {df.shape[0]} pessoas')
print()
print(f'PVSE_A  Exclusão Produtiva Analfabeta    : {pvse_a} pessoas')
print(f'PVSE_B  Maternidade Vulnerável Severa    : {pvse_b} pessoas')
print(f'PVSE_C  Idoso Desprotegido               : {pvse_c} pessoas')
print()

# distribuição territorial dos três perfis combinados — proxy nom_localidade_fam
mask_pvse_total = (
    (mask_15_59 & mask_analfabeto & mask_sem_renda) |
    (mask_analfabeto & mask_feminino & mask_sem_pensao) |
    (mask_idoso_analfabeto & mask_renda_baixa & mask_sem_aposent & mask_nao_pbf)
)

pvse_por_local = (
    df[mask_pvse_total]['d.nom_localidade_fam']
    .value_counts()
    .reset_index()
)
pvse_por_local.columns = ['localidade', 'qtd_pvse']

print('Top 10 localidades com maior concentração PVSE:')
print(pvse_por_local.head(10).to_string(index=False))